<a href="https://colab.research.google.com/github/zehzin/Lista-de-Filmes/blob/master/projeto_livro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# ==============================================================================
# CÉLULA 1: INSTALAÇÃO DE DEPENDÊNCIAS
# Objetivo: Instalar pacotes do sistema (Pandoc/XeTeX) e bibliotecas Python.
# ==============================================================================

# Instala dependências do sistema
!apt-get update -qq && apt-get install -y zstd pandoc texlive-xetex -qq

# Atualiza e instala pacotes Python
!pip install --upgrade pillow==11.3.0 --quiet
!pip install ollama pdfplumber fpdf PyPDF2 requests --quiet

print("✅ Dependências instaladas com sucesso!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
✅ Dependências instaladas com sucesso!


In [24]:
# ==============================================================================
# CÉLULA 2: INSTALAR E INICIAR O OLLAMA
# Objetivo: Baixar o instalador oficial e rodar o servidor em segundo plano.
# ==============================================================================

import time
import requests

# Instala o Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Inicia o servidor Ollama em segundo plano usando bash (nohup)
print("\nIniciando o servidor Ollama em background...")
!nohup ollama serve > ollama.log 2>&1 &

# Aguarda o servidor subir e verifica a conexão
time.sleep(5)

try:
    response = requests.get("http://127.0.0.1:11434/")
    if response.status_code == 200:
        print("✅ Servidor Ollama pronto e rodando!")
except requests.exceptions.ConnectionError:
    print("⚠️ Erro: O servidor Ollama não iniciou corretamente. Verifique os logs.")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Iniciando o servidor Ollama em background...
✅ Servidor Ollama pronto e rodando!


In [25]:
# ==============================================================================
# CÉLULA 3: BAIXAR O MODELO DE IA
# Objetivo: Fazer o download do modelo Llama 3.1 (ou outro de sua escolha).
# ==============================================================================

# Altere para o modelo desejado (ex: llama3, mistral, phi3)
modelo = "llama3.1:8b"
print(f"Baixando o modelo {modelo}... Isso pode levar alguns minutos.")

!ollama pull {modelo}
print("✅ Modelo baixado com sucesso!")

Baixando o modelo llama3.1:8b... Isso pode levar alguns minutos.

✅ Modelo baixado com sucesso!


In [27]:
# ==============================================================================
# CÉLULA 4: UPLOAD DE ARQUIVOS DE REFERÊNCIA
# Objetivo: Criar a pasta 'referencias' e permitir o envio de PDFs ou TXTs.
# ==============================================================================

from google.colab import files
import os
import shutil

os.makedirs("referencias", exist_ok=True)

print("Selecione os arquivos de referência que deseja usar (PDF, TXT, MD).")
print("Caso não queira usar referências, basta cancelar o upload.")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"referencias/{filename}")
    print(f"✅ Arquivo '{filename}' salvo em 'referencias/'.")

Selecione os arquivos de referência que deseja usar (PDF, TXT, MD).
Caso não queira usar referências, basta cancelar o upload.


KeyboardInterrupt: 

In [28]:
%%writefile gerar_livro_colab.py
# ==============================================================================
# CÉLULA 5: CRIAR O SCRIPT GERADOR DO LIVRO
# Objetivo: Salvar o código Python que fará a leitura, geração e exportação.
# (Atenção: O comando %%writefile acima DEVE ser a linha 1 desta célula)
# ==============================================================================

import os
import re
import sys
import argparse
import warnings
from pathlib import Path
from datetime import datetime
import ollama

# Suprime avisos
warnings.filterwarnings("ignore", category=UserWarning)

# Dependências Opcionais
try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False

try:
    from fpdf import FPDF
    FPDF_AVAILABLE = True
except ImportError:
    FPDF_AVAILABLE = False

# CONFIGURAÇÕES
PASTA_REFERENCIAS = "referencias"
PASTA_SAIDA = "output"
ARQUIVO_SAIDA_MD = "livro_gerado.md"
ARQUIVO_SAIDA_PDF = "livro_gerado.pdf"
MODELO_OLLAMA = "llama3.1:8b"

def ler_arquivo_texto(caminho):
    try:
        with open(caminho, 'r', encoding='utf-8') as f:
            return f.read()
    except UnicodeDecodeError:
        with open(caminho, 'r', encoding='latin-1') as f:
            return f.read()

def ler_arquivo_pdf(caminho):
    texto = ""
    if PDFPLUMBER_AVAILABLE:
        try:
            with pdfplumber.open(caminho) as pdf:
                for pagina in pdf.pages:
                    try:
                        t = pagina.extract_text()
                        if t: texto += t
                    except:
                        continue
        except Exception as e:
            print(f"Erro ao ler PDF: {e}")
    return texto

def carregar_referencias():
    referencias = []
    pasta = Path(PASTA_REFERENCIAS)
    if not pasta.exists():
        return referencias

    for arquivo in pasta.iterdir():
        if arquivo.is_file():
            try:
                if arquivo.suffix.lower() in ['.txt', '.md']:
                    texto = ler_arquivo_texto(arquivo)
                elif arquivo.suffix.lower() == '.pdf':
                    texto = ler_arquivo_pdf(arquivo)
                else:
                    continue

                if texto.strip():
                    referencias.append({'nome': arquivo.name, 'texto': texto})
                    print(f" ✓ {arquivo.name} carregado ({len(texto)} caracteres)")
            except Exception as e:
                print(f" ✗ Erro em {arquivo.name}: {e}")
    return referencias

def chamar_ollama(prompt, modelo, temperatura=0.7, max_tokens=8192):
    try:
        response = ollama.chat(
            model=modelo,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": temperatura, "num_predict": max_tokens}
        )
        return response["message"]["content"].strip()
    except Exception as e:
        print(f" ✗ ERRO no modelo {modelo}: {e}")
        return ""

def gerar_sumario(tema, referencias_texto):
    refs = referencias_texto[:6000] if referencias_texto else ""
    prompt = f"""Crie um sumário detalhado para um livro sobre "{tema}".
O livro deve ter de 10 a 15 capítulos.
Responda APENAS com uma lista numerada, UM CAPÍTULO POR LINHA, no formato:
1. Título do capítulo 1
2. Título do capítulo 2
Não inclua texto introdutório ou explicações. Apenas a lista."""

    resposta = chamar_ollama(prompt, MODELO_OLLAMA)
    capitulos = [re.sub(r'^\d+\.\s*', '', linha.strip()) for linha in resposta.split('\n') if re.match(r'^\d+\.', linha.strip())]

    return capitulos if capitulos else ["Introdução", "Desenvolvimento", "Conclusão"]

def gerar_capitulo(titulo, tema, referencias_texto, numero, total):
    refs = referencias_texto[:20000] if referencias_texto else ""
    prompt = f"""Escreva o capítulo "{titulo}" para um livro sobre "{tema}".
Estruture o capítulo com:
- Uma introdução que contextualiza o tema.
- De 3 a 5 subseções com títulos próprios (use ## para cada subtítulo).
- Uma conclusão que amarre as ideias.
Use as referências abaixo para fundamentar cada parte. Sempre que citar, use (Autor, Ano).
Referências: {refs}
O capítulo deve ter entre 3000 e 5000 palavras, bem escrito e coeso."""

    percentual = (numero / total) * 100
    barra = f"[{'#' * int(percentual//5)}{'.' * (20 - int(percentual//5))}]"
    print(f" Gerando capítulo {numero}/{total} ({percentual:.1f}%) {barra}")

    conteudo = chamar_ollama(prompt, MODELO_OLLAMA, max_tokens=8192)
    return conteudo if conteudo else f"(Conteúdo do capítulo '{titulo}' não gerado.)"

def extrair_citacoes(texto):
    padrao = r'\(([^)]+?,\s*\d{4}[^)]*?)\)'
    return list(set([c.strip() for c in re.findall(padrao, texto)]))

def montar_livro(tema, capitulos, referencias_texto):
    livro = f"# {tema}\n\n*Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M')}*\n\n## Sumário\n\n"
    for i, cap in enumerate(capitulos, 1):
        livro += f"{i}. {cap}\n"
    livro += "\n---\n\n"

    todas_citacoes = []
    total = len(capitulos)

    for i, cap in enumerate(capitulos, 1):
        livro += f"## Capítulo {i}: {cap}\n\n"
        conteudo = gerar_capitulo(cap, tema, referencias_texto, i, total)
        livro += conteudo + "\n\n---\n\n"
        todas_citacoes.extend(extrair_citacoes(conteudo))

    if todas_citacoes:
        livro += "# Referências\n\n"
        for i, cit in enumerate(sorted(set(todas_citacoes)), 1):
            livro += f"{i}. {cit}\n"

    return livro

def converter_para_pdf(md_path, pdf_path):
    try:
        import subprocess
        subprocess.run(["pandoc", str(md_path), "-o", str(pdf_path), "--pdf-engine=xelatex", "-V", "geometry:margin=2.5cm", "--toc"], check=True)
        print(f"✓ PDF gerado com pandoc: {pdf_path}")
        return True
    except:
        print("Pandoc falhou. Tentando FPDF...")
        if FPDF_AVAILABLE:
            try:
                with open(md_path, 'r', encoding='utf-8') as f: texto = f.read()
                pdf = FPDF()
                pdf.add_page()
                pdf.set_font("Arial", size=12)
                for linha in texto.split('\n'):
                    if linha.startswith('#'):
                        pdf.set_font("Arial", 'B', 14)
                        pdf.cell(0, 10, linha.replace('#', '').strip(), ln=True)
                        pdf.set_font("Arial", size=12)
                    else:
                        pdf.multi_cell(0, 8, linha)
                pdf.output(str(pdf_path))
                print("✓ PDF gerado com FPDF.")
                return True
            except Exception as e:
                print(f"Erro FPDF: {e}")
    return False

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--tema", required=True)
    args = parser.parse_args()

    print("=" * 60)
    print(" GERADOR DE LIVRO COM IA (Ollama)")
    print("=" * 60)

    referencias = carregar_referencias()
    texto_refs = "\n\n".join([r['texto'] for r in referencias])[:20000] if referencias else ""

    print("\n[1/4] Gerando sumário...")
    capitulos = gerar_sumario(args.tema, texto_refs)

    print("\n[2/4] Gerando capítulos...")
    livro_md = montar_livro(args.tema, capitulos, texto_refs)

    print("\n[3/4] Salvando Markdown...")
    Path(PASTA_SAIDA).mkdir(exist_ok=True)
    md_path = Path(PASTA_SAIDA) / ARQUIVO_SAIDA_MD
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(livro_md)

    print("\n[4/4] Convertendo para PDF...")
    converter_para_pdf(md_path, Path(PASTA_SAIDA) / ARQUIVO_SAIDA_PDF)
    print("\n✅ Processo concluído!")

if __name__ == "__main__":
    main()

Overwriting gerar_livro_colab.py


In [29]:
# Célula 5.5: Verificar e reiniciar o servidor Ollama (evita erros de conexão)
import subprocess
import time
import requests
import os

print("Verificando se o Ollama está rodando...")

def check_ollama():
    try:
        response = requests.get("http://localhost:11434/api/tags")
        return response.status_code == 200
    except:
        return False

# Se não estiver rodando, inicia
if not check_ollama():
    print("Iniciando servidor Ollama...")
    # Mata qualquer processo anterior
    !pkill -f ollama 2>/dev/null
    # Define variáveis de ambiente
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    # Inicia o servidor em background
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(10)
else:
    print("✅ Ollama já está rodando.")

# Verifica novamente
if check_ollama():
    print("✅ Servidor Ollama pronto!")
    # Lista os modelos disponíveis
    !ollama list
else:
    print("❌ Falha ao iniciar o Ollama. Execute manualmente:")
    print("!ollama serve &")

Verificando se o Ollama está rodando...
Iniciando servidor Ollama...
^C
✅ Servidor Ollama pronto!
NAME           ID              SIZE      MODIFIED       
llama3.1:8b    46e0c10c039e    4.9 GB    31 seconds ago    


In [30]:
# ==============================================================================
# CÉLULA 6: EXECUTAR O GERADOR
# Objetivo: Pedir o tema do livro e iniciar o processo de geração.
# ==============================================================================

# Pede o tema para o usuário
tema = input("Digite o tema do livro (Ex: 'A História da Inteligência Artificial'): ")

# Roda o script passando o tema como argumento
!python gerar_livro_colab.py --tema "{tema}"

Digite o tema do livro (Ex: 'A História da Inteligência Artificial'): O Arco-iris das Emoções
 GERADOR DE LIVRO COM IA (Ollama)
 ✓ o_cerebro_e_a_inteligencia_emocional.pdf carregado (117010 caracteres)
 ✓ as-48-leis-do-poder-robert-greene.pdf carregado (311470 caracteres)
 ✓ PDF-M2U3-Inteligência-emocional-_FORPRES-2023.pdf carregado (43405 caracteres)
 ✓ as-armas-da-persuasao.pdf carregado (712345 caracteres)
 ✓ arte-da-guerra.pdf carregado (63502 caracteres)
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontB

In [31]:
# ==============================================================================
# CÉLULA 7: BAIXAR OS ARQUIVOS GERADOS
# Objetivo: Fazer o download do PDF e do arquivo Markdown para o seu computador.
# ==============================================================================

from google.colab import files
import os

pdf_path = 'output/livro_gerado.pdf'
md_path = 'output/livro_gerado.md'

if os.path.exists(pdf_path):
    files.download(pdf_path)
else:
    print("⚠️ Arquivo PDF não encontrado. Pode ter ocorrido um erro na geração.")

if os.path.exists(md_path):
    files.download(md_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>